# Closed-Source Models — Base Benchmark Generation

This notebook reproduces the **generation stage** of the closed-source model experiments on the Turkish reasoning base benchmark.

Models used in the original experiment:

- GPT-4o-mini
- GPT-4o
- Claude Sonnet 4
- Gemini 2.5 Flash

For each question, the model is asked to produce:

- reasoning,
- a final answer,
- a confidence score between 0 and 100.

This notebook handles **generation only**.

Model evaluation is intentionally kept in a separate notebook.

## Important safety and reproducibility notes

### API calls

Generation is **disabled by default**.

The configuration variable:

```python
RUN_GENERATION = False

## 1. Install dependencies

Only lightweight API client libraries are required.

A GPU is **not required** for this notebook because model inference is performed
through external APIs.

In [25]:
%pip install -q openai anthropic google-genai pandas tqdm

In [26]:
from pathlib import Path
import json
import os
import sys
import time

import pandas as pd
from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

Python: 3.13.15
pandas: 2.2.3


## 2. Locate the project

The notebook supports two common workflows:

1. running from a cloned GitHub repository,
2. running from a copy stored in Google Drive.

The public GitHub version defaults to repository mode.

In [27]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

USE_GOOGLE_DRIVE = True

DRIVE_PROJECT_PATH = Path(
    "/content/drive/MyDrive/turkish-llm-reasoning"
)


if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        PROJECT_ROOT = DRIVE_PROJECT_PATH

    except ImportError:
        raise RuntimeError(
            "Google Drive mounting is only available in Google Colab."
        )

else:
    current_path = Path.cwd().resolve()
    candidates = [current_path, *current_path.parents]

    PROJECT_ROOT = None

    for candidate in candidates:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            PROJECT_ROOT = candidate
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Project root could not be located. "
            "Run the notebook from inside the repository "
            "or enable Google Drive mode."
        )


print("Project root:")
print(PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root:
/content/drive/MyDrive/turkish-llm-reasoning


## 3. Dataset configuration and reproducibility

This notebook supports different versions of the benchmark without hard-coding
the dataset size throughout the code.

Two dataset profiles are defined:

- **original** — the snapshot used in the original closed-model experiment.
- **current** — an optional evolving version of the benchmark for future experiments.

The `original` profile includes strict size checks so that the historical
experiment cannot accidentally be reproduced with a different dataset.

The `current` profile does not assume a fixed number of questions.

A SHA-256 hash of the dataset file is also recorded so that experiment outputs
can later be linked to the exact dataset version that produced them.

In [28]:
# ============================================================
# DATASET CONFIGURATION
# ============================================================

DATASET_PROFILE = "smoke"

DATASET_PROFILES = {
    "original": {
        "path": PROJECT_ROOT / "data" / "base" / "base_questions.csv",
        "expected_raw_rows": 489,
        "expected_usable_rows": 488,
        "description": "Original closed-model experiment snapshot",
    },

    "current": {
        "path": PROJECT_ROOT / "data" / "base" / "base_questions_current.csv",
        "expected_raw_rows": None,
        "expected_usable_rows": None,
        "description": "Current evolving benchmark",
    },

    "smoke": {
    "path": PROJECT_ROOT / "data" / "base" / "smoke_base_v1.csv",
    "expected_raw_rows": 5,
    "expected_usable_rows": 5,
    "description": "Small smoke-test dataset for pipeline validation",
    },
}


if DATASET_PROFILE not in DATASET_PROFILES:
    raise ValueError(
        f"Unknown dataset profile: {DATASET_PROFILE}"
    )


DATASET_CONFIG = DATASET_PROFILES[DATASET_PROFILE]
BASE_DATA_PATH = DATASET_CONFIG["path"]


print("Dataset profile:")
print(f"  Name:        {DATASET_PROFILE}")
print(f"  Description: {DATASET_CONFIG['description']}")
print(f"  Path:        {BASE_DATA_PATH}")

Dataset profile:
  Name:        smoke
  Description: Small smoke-test dataset for pipeline validation
  Path:        /content/drive/MyDrive/turkish-llm-reasoning/data/base/smoke_base_v1.csv


In [29]:
import hashlib


def calculate_file_sha256(filepath):
    """
    Calculate the SHA-256 hash of a file.

    The hash makes it possible to identify the exact dataset
    snapshot used in an experiment.
    """

    sha256 = hashlib.sha256()

    with open(filepath, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

In [30]:
if not BASE_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {BASE_DATA_PATH}"
    )


raw_df = pd.read_csv(BASE_DATA_PATH)


REQUIRED_FIELDS = [
    "question_id",
    "category",
    "difficulty",
    "context",
    "question",
    "gold_evidence",
    "gold_answer",
]


missing_columns = [
    column
    for column in REQUIRED_FIELDS
    if column not in raw_df.columns
]


if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


df = (
    raw_df
    .dropna(
        subset=[
            "gold_answer",
            "gold_evidence",
            "context",
            "question",
        ]
    )
    .copy()
)


DATASET_SHA256 = calculate_file_sha256(
    BASE_DATA_PATH
)


print(f"Raw dataset:     {len(raw_df)} rows")
print(f"Usable dataset:  {len(df)} rows")
print(f"Excluded:        {len(raw_df) - len(df)} rows")
print(f"Dataset SHA-256: {DATASET_SHA256}")

Raw dataset:     5 rows
Usable dataset:  5 rows
Excluded:        0 rows
Dataset SHA-256: 20d742f0441f593d59cc929c7c8d7b9cc9c1efbd10b8224dbc4519935e8e61f9


In [31]:
expected_raw_rows = DATASET_CONFIG[
    "expected_raw_rows"
]

expected_usable_rows = DATASET_CONFIG[
    "expected_usable_rows"
]


if (
    expected_raw_rows is not None
    and len(raw_df) != expected_raw_rows
):
    raise ValueError(
        f"Dataset profile '{DATASET_PROFILE}' expects "
        f"{expected_raw_rows} raw rows, "
        f"but {len(raw_df)} were found."
    )


if (
    expected_usable_rows is not None
    and len(df) != expected_usable_rows
):
    raise ValueError(
        f"Dataset profile '{DATASET_PROFILE}' expects "
        f"{expected_usable_rows} usable rows, "
        f"but {len(df)} were found."
    )


if not raw_df["question_id"].is_unique:
    raise ValueError(
        "question_id values must be unique."
    )


print("✓ Dataset validation passed.")

✓ Dataset validation passed.


### Historical dataset note

The original closed-model experiment used a benchmark snapshot containing
**489 records**.

One record was missing the reference answer and supporting evidence required by
the experiment pipeline. As a result, **488 records** were used for generation.

These numbers describe the historical dataset snapshot only.

Future versions of the benchmark may contain a different number of questions.

## 4. Experiment configuration

The configuration below defines the model and generation settings.

The default setup corresponds to the original closed-model experiment.

Generation remains disabled unless `RUN_GENERATION` is explicitly changed to
`True`.


Using a different model ID creates a new experiment and should not be treated
as an exact reproduction of the historical run.

In [32]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

RUN_GENERATION = True

SELECTED_MODEL = "gpt-4o-mini"

RUN_LABEL = "smoke_v1"


# ============================================================
# MODEL REGISTRY
# ============================================================

MODEL_REGISTRY = {

    "gpt-4o-mini": {
        "provider": "openai",
        "model_id": "gpt-4o-mini",
        "temperature": 0,
        "max_tokens": 2048,
        "historical": True,
    },

    "gpt-4o": {
        "provider": "openai",
        "model_id": "gpt-4o",
        "temperature": 0,
        "max_tokens": 2048,
        "historical": True,
    },

    "claude-sonnet-4": {
        "provider": "anthropic",
        "model_id": "claude-sonnet-4-20250514",
        "temperature": None,
        "max_tokens": 2048,
        "historical": True,
    },

    "gemini-2.5-flash": {
        "provider": "google",
        "model_id": "gemini-2.5-flash",
        "temperature": None,
        "max_tokens": None,
        "historical": True,
    },
}


if SELECTED_MODEL not in MODEL_REGISTRY:

    raise ValueError(
        f"Unknown model: {SELECTED_MODEL}. "
        f"Available models: {list(MODEL_REGISTRY)}"
    )


MODEL_CONFIG = MODEL_REGISTRY[
    SELECTED_MODEL
]

PROVIDER = MODEL_CONFIG[
    "provider"
]

MODEL_ID = MODEL_CONFIG[
    "model_id"
]

TEMPERATURE = MODEL_CONFIG[
    "temperature"
]

MAX_TOKENS = MODEL_CONFIG[
    "max_tokens"
]

IS_HISTORICAL_MODEL = MODEL_CONFIG[
    "historical"
]


print("Selected experiment:")
print(f"  Dataset profile: {DATASET_PROFILE}")
print(f"  Dataset rows:    {len(df)}")
print(f"  Model label:     {SELECTED_MODEL}")
print(f"  Provider:        {PROVIDER}")
print(f"  API model ID:    {MODEL_ID}")
print(f"  Historical:      {IS_HISTORICAL_MODEL}")
print(f"  Temperature:     {TEMPERATURE}")
print(f"  Max tokens:      {MAX_TOKENS}")
print(f"  Generation:      {RUN_GENERATION}")

Selected experiment:
  Dataset profile: smoke
  Dataset rows:    5
  Model label:     gpt-4o-mini
  Provider:        openai
  API model ID:    gpt-4o-mini
  Historical:      True
  Temperature:     0
  Max tokens:      2048
  Generation:      True


## 5. Prompt

The prompt below preserves the structure used in the original experiment.

For each benchmark item, the model receives a context and a question and is
asked to return a JSON object containing:

- `reasoning`
- `answer`
- `confidence`

The prompt is intentionally kept stable because changing the prompt would
change the experimental condition.

In [33]:
SYSTEM_PROMPT = """Sen bir problem çözme asistanısın. Sana verilen bağlam (context) ve soruyu (question) analiz edeceksin.

GÖREV:
1. Bağlamdaki bilgileri kullanarak adım adım düşün.
2. Her adımda hangi bilgiyi kullandığını açıkça belirt.
3. Sonuca ulaş ve cevabını ver.

ÇIKTI FORMATI (JSON):
{
    "reasoning": "Adım adım düşünce sürecin. Her adımda bağlamdan hangi bilgiyi kullandığını yaz.",
    "answer": "Nihai cevabın (kısa ve net)",
    "confidence": 0-100 arası bir sayı (cevabından ne kadar eminsin)
}

ÖNEMLİ:
- Sadece bağlamda verilen bilgileri kullan, dışarıdan bilgi ekleme.
- Cevabın SADECE geçerli bir JSON objesi olmalı, başka bir şey yazma.
- Türkçe yanıt ver."""

In [34]:
def build_user_prompt(context, question):
    return f"""BAĞLAM:
{context}

SORU:
{question}

Yukarıdaki bağlamı kullanarak soruyu cevapla. Yanıtını JSON formatında ver."""

## 6. API credentials

API keys must never be written directly into the notebook or committed to the
repository.

When generation is enabled, credentials can be read from:

1. environment variables,
2. Google Colab Secrets.

Expected secret names:

- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GOOGLE_API_KEY`

No API request is made in this section.

In [35]:
def get_secret(name):
    """
    Read a secret from an environment variable or,
    when available, Google Colab Secrets.
    """

    value = os.getenv(name)

    if value:
        return value

    try:
        from google.colab import userdata
        return userdata.get(name)

    except Exception:
        return None

## 7. Configuration summary

The following cell prints the current dataset and model configuration before
any generation code is enabled.

This makes it easier to verify the experimental setup before sending API
requests.

In [36]:
print("=" * 60)
print("CLOSED MODELS — BASE GENERATION")
print("=" * 60)

print("\nDataset")
print(f"  Profile:        {DATASET_PROFILE}")
print(f"  Raw rows:       {len(raw_df)}")
print(f"  Usable rows:    {len(df)}")
print(f"  SHA-256:        {DATASET_SHA256}")

print("\nModel")
print(f"  Selected model: {SELECTED_MODEL}")
print(f"  Provider:       {PROVIDER}")
print(f"  Model ID:       {MODEL_ID}")

print("\nGeneration")
print(f"  Temperature:    {TEMPERATURE}")
print(f"  Max tokens:     {MAX_TOKENS}")
print(f"  Enabled:        {RUN_GENERATION}")

print("\nNo API request has been sent.")

CLOSED MODELS — BASE GENERATION

Dataset
  Profile:        smoke
  Raw rows:       5
  Usable rows:    5
  SHA-256:        20d742f0441f593d59cc929c7c8d7b9cc9c1efbd10b8224dbc4519935e8e61f9

Model
  Selected model: gpt-4o-mini
  Provider:       openai
  Model ID:       gpt-4o-mini

Generation
  Temperature:    0
  Max tokens:     2048
  Enabled:        True

No API request has been sent.


## 8. Generation and checkpointing

This section contains the model generation pipeline.

Generation is disabled by default.

When enabled, the pipeline:

- initializes only the selected model provider,
- sends each usable benchmark question to the selected model,
- preserves the raw model response,
- parses the structured JSON response,
- saves progress periodically,
- resumes from an existing checkpoint,
- records dataset, prompt, and model metadata.

The raw response is preserved even when JSON parsing fails so that failed
records can be inspected or recovered later without repeating the API call.

In [37]:
# ============================================================
# OUTPUT AND CHECKPOINT CONFIGURATION
# ============================================================

CHECKPOINT_EVERY = 5
REQUEST_DELAY_SECONDS = 0.5


def safe_name(value):
    return (
        str(value)
        .strip()
        .lower()
        .replace("/", "-")
        .replace(" ", "-")
    )


RUN_NAME = (
    f"{safe_name(DATASET_PROFILE)}__"
    f"{safe_name(SELECTED_MODEL)}__"
    f"{safe_name(RUN_LABEL)}"
)


RAW_OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "raw"
    / "base"
)

CHECKPOINT_DIR = (
    RAW_OUTPUT_DIR
    / "checkpoints"
)


CHECKPOINT_FILE = (
    CHECKPOINT_DIR
    / f"{RUN_NAME}.json"
)

FINAL_OUTPUT_FILE = (
    RAW_OUTPUT_DIR
    / f"{RUN_NAME}.json"
)

METADATA_FILE = (
    RAW_OUTPUT_DIR
    / f"{RUN_NAME}.metadata.json"
)


print("Run name:")
print(RUN_NAME)

print("\nCheckpoint:")
print(CHECKPOINT_FILE)

print("\nFinal output:")
print(FINAL_OUTPUT_FILE)

Run name:
smoke__gpt-4o-mini__smoke_v1

Checkpoint:
/content/drive/MyDrive/turkish-llm-reasoning/results/raw/base/checkpoints/smoke__gpt-4o-mini__smoke_v1.json

Final output:
/content/drive/MyDrive/turkish-llm-reasoning/results/raw/base/smoke__gpt-4o-mini__smoke_v1.json


### Prompt fingerprint

In addition to the dataset hash, the generation pipeline records a hash of the
prompt configuration.

This helps distinguish results generated with different prompt versions.

In [38]:
PROMPT_REFERENCE = (
    SYSTEM_PROMPT
    + "\n---USER-PROMPT---\n"
    + build_user_prompt(
        "{context}",
        "{question}",
    )
)


PROMPT_SHA256 = hashlib.sha256(
    PROMPT_REFERENCE.encode("utf-8")
).hexdigest()


print("Prompt SHA-256:")
print(PROMPT_SHA256)

Prompt SHA-256:
1663da14c062cef14f02ba36cab60ef772b1e948f1ba9b8144be7f35d859e25a


## 9. JSON response parser

Models are instructed to return a JSON object.

The parser below accepts:

1. direct JSON,
2. JSON inside a Markdown code block,
3. a JSON object embedded inside surrounding text.

If parsing fails, the raw response is still preserved.

In [39]:
import re


def extract_json_from_response(content):
    """
    Extract a JSON object from a model response.

    Returns:
        dict or None
    """

    if content is None:
        return None

    content = str(content).strip()

    if not content:
        return None


    # --------------------------------------------------------
    # 1. Direct JSON
    # --------------------------------------------------------

    try:
        parsed = json.loads(content)

        if isinstance(parsed, dict):
            return parsed

    except json.JSONDecodeError:
        pass


    # --------------------------------------------------------
    # 2. Markdown code block
    # --------------------------------------------------------

    code_blocks = re.findall(
        r"```(?:json)?\s*(.*?)```",
        content,
        flags=re.DOTALL | re.IGNORECASE,
    )

    for block in code_blocks:

        try:
            parsed = json.loads(block.strip())

            if isinstance(parsed, dict):
                return parsed

        except json.JSONDecodeError:
            continue


    # --------------------------------------------------------
    # 3. JSON object inside surrounding text
    # --------------------------------------------------------

    start = content.find("{")
    end = content.rfind("}")

    if start != -1 and end > start:

        candidate = content[start:end + 1]

        try:
            parsed = json.loads(candidate)

            if isinstance(parsed, dict):
                return parsed

        except json.JSONDecodeError:
            pass


    return None

## 10. Provider clients

Only the provider required by the selected model is initialized.

API credentials are loaded only when generation is explicitly enabled.

Adding another model from an already supported provider normally requires only
adding a new entry to `MODEL_REGISTRY`.

Adding a completely new provider requires implementing one additional provider
adapter.

In [40]:
def create_provider_client():
    """
    Create the API client required by the selected model.
    """

    if PROVIDER == "openai":

        api_key = get_secret(
            "OPENAI_API_KEY"
        )

        if not api_key:
            raise RuntimeError(
                "OPENAI_API_KEY was not found."
            )

        from openai import OpenAI

        return OpenAI(
            api_key=api_key
        )


    elif PROVIDER == "anthropic":

        api_key = get_secret(
            "ANTHROPIC_API_KEY"
        )

        if not api_key:
            raise RuntimeError(
                "ANTHROPIC_API_KEY was not found."
            )

        from anthropic import Anthropic

        return Anthropic(
            api_key=api_key
        )


    elif PROVIDER == "google":

        api_key = get_secret(
            "GOOGLE_API_KEY"
        )

        if not api_key:
            raise RuntimeError(
                "GOOGLE_API_KEY was not found."
            )

        from google import genai

        return genai.Client(
            api_key=api_key
        )


    else:
        raise NotImplementedError(
            f"Provider not supported: {PROVIDER}"
        )

## 11. Single-response generation

Provider-specific API calls are isolated in a single function.

The returned record always follows the same internal structure regardless of
the provider.

The original experiment used slightly different request styles across
providers. This notebook preserves that distinction rather than silently
forcing every provider into the same API format.

In [41]:
def generate_single_response(
    client,
    context,
    question,
):
    """
    Generate one model response.

    Raw output is always preserved when available.
    """

    user_prompt = build_user_prompt(
        context,
        question,
    )

    raw_response = None


    try:

        # ====================================================
        # OpenAI
        # ====================================================

        if PROVIDER == "openai":

            request_kwargs = {
                "model": MODEL_ID,

                "messages": [
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": user_prompt,
                    },
                ],

                "response_format": {
                    "type": "json_object"
                },
            }


            if TEMPERATURE is not None:
                request_kwargs["temperature"] = TEMPERATURE

            if MAX_TOKENS is not None:
                request_kwargs["max_tokens"] = MAX_TOKENS


            response = (
                client
                .chat
                .completions
                .create(**request_kwargs)
            )

            raw_response = (
                response
                .choices[0]
                .message
                .content
            )


        # ====================================================
        # Anthropic
        # ====================================================

        elif PROVIDER == "anthropic":

            request_kwargs = {
                "model": MODEL_ID,

                "messages": [
                    {
                        "role": "user",

                        "content": (
                            SYSTEM_PROMPT
                            + "\n\n"
                            + user_prompt
                        ),
                    }
                ],
            }


            if MAX_TOKENS is not None:
                request_kwargs["max_tokens"] = MAX_TOKENS

            if TEMPERATURE is not None:
                request_kwargs["temperature"] = TEMPERATURE


            response = (
                client
                .messages
                .create(**request_kwargs)
            )


            text_parts = [
                block.text

                for block in response.content

                if getattr(
                    block,
                    "type",
                    None,
                ) == "text"
            ]

            raw_response = "\n".join(
                text_parts
            )


        # ====================================================
        # Google Gemini
        # ====================================================

        elif PROVIDER == "google":

            full_prompt = (
                SYSTEM_PROMPT
                + "\n\n"
                + user_prompt
            )

            generation_settings = {}


            if TEMPERATURE is not None:
                generation_settings[
                    "temperature"
                ] = TEMPERATURE

            if MAX_TOKENS is not None:
                generation_settings[
                    "max_output_tokens"
                ] = MAX_TOKENS


            if generation_settings:

                from google.genai import types

                response = (
                    client
                    .models
                    .generate_content(
                        model=MODEL_ID,
                        contents=full_prompt,
                        config=types.GenerateContentConfig(
                            **generation_settings
                        ),
                    )
                )

            else:

                response = (
                    client
                    .models
                    .generate_content(
                        model=MODEL_ID,
                        contents=full_prompt,
                    )
                )


            raw_response = response.text


        else:

            raise NotImplementedError(
                f"Provider not supported: {PROVIDER}"
            )


    except Exception as exc:

        return {
            "reasoning": None,
            "answer": None,
            "confidence": None,
            "raw_response": raw_response,

            "error": (
                f"{type(exc).__name__}: {exc}"
            ),
        }


    parsed = extract_json_from_response(
        raw_response
    )


    if parsed is None:

        return {
            "reasoning": None,
            "answer": None,
            "confidence": None,
            "raw_response": raw_response,
            "error": "JSON parsing failed",
        }


    return {
        "reasoning": parsed.get("reasoning"),
        "answer": parsed.get("answer"),
        "confidence": parsed.get("confidence"),
        "raw_response": raw_response,
        "error": None,
    }

## 12. Run metadata

Each generation run stores enough metadata to identify the experimental
configuration without relying on the notebook filename.

This includes the exact dataset hash, prompt hash, model ID, provider, and
generation settings.

In [42]:
from datetime import datetime, timezone


def utc_now():
    return (
        datetime
        .now(timezone.utc)
        .isoformat()
    )


def build_run_metadata():

    return {


        "experiment_type": "closed_models_base",
        "run_name": RUN_NAME,

        "dataset_profile": (
            DATASET_PROFILE
        ),

        "dataset_path": str(
            BASE_DATA_PATH
        ),

        "dataset_sha256": (
            DATASET_SHA256
        ),

        "raw_dataset_rows": (
            len(raw_df)
        ),

        "usable_dataset_rows": (
            len(df)
        ),

        "prompt_sha256": (
            PROMPT_SHA256
        ),

        "model_label": (
            SELECTED_MODEL
        ),

        "model_id": (
            MODEL_ID
        ),

        "provider": (
            PROVIDER
        ),

        "historical_model": (
            IS_HISTORICAL_MODEL
        ),

        "temperature": (
            TEMPERATURE
        ),

        "max_tokens": (
            MAX_TOKENS
        ),
    }

## 13. Checkpoint helpers

Checkpoint files allow long experiments to continue after a runtime
interruption.

Before resuming, the notebook verifies that the checkpoint belongs to the same
dataset, prompt, and model configuration.

This prevents an old checkpoint from being accidentally mixed with a different
experiment.

In [43]:
def save_json_atomic(
    data,
    filepath,
):
    """
    Save JSON through a temporary file to reduce the risk
    of checkpoint corruption.
    """

    filepath = Path(filepath)

    filepath.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_file = filepath.with_suffix(
        filepath.suffix + ".tmp"
    )

    with open(
        temporary_file,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    os.replace(
        temporary_file,
        filepath,
    )

In [44]:
def load_checkpoint(
    checkpoint_file,
    current_metadata,
):
    """
    Load and validate an existing checkpoint.
    """

    checkpoint_file = Path(
        checkpoint_file
    )


    if not checkpoint_file.exists():

        return [], None


    with open(
        checkpoint_file,
        "r",
        encoding="utf-8",
    ) as file:

        checkpoint = json.load(file)


    saved_metadata = checkpoint[
        "metadata"
    ]

    records = checkpoint[
        "records"
    ]


    fields_to_validate = [
        "dataset_sha256",
        "prompt_sha256",
        "model_id",
        "provider",
        "temperature",
        "max_tokens"
    ]


    for field in fields_to_validate:

        if (
            saved_metadata.get(field)
            != current_metadata.get(field)
        ):

            raise ValueError(
                "Checkpoint configuration mismatch "
                f"for '{field}'."
            )


    print(
        "✓ Existing checkpoint loaded:"
        f" {len(records)} records"
    )


    return (
        records,
        saved_metadata.get(
            "created_at"
        ),
    )

## 14. Run generation

The function below processes only questions that are not already present in the
checkpoint.

Errors are preserved as experiment records rather than being silently removed.

Automatic retry of failed requests is intentionally not performed here because
a retry may create additional API cost. Failed records can be reviewed
separately.

In [45]:
def run_generation():

    client = create_provider_client()

    metadata = build_run_metadata()

    records, created_at = load_checkpoint(
        CHECKPOINT_FILE,
        metadata,
    )


    if created_at is None:
        created_at = utc_now()


    processed_ids = {
        str(record["question_id"])
        for record in records
    }


    remaining_df = df[
        ~df["question_id"]
        .astype(str)
        .isin(processed_ids)
    ]


    print(
        f"Already processed: "
        f"{len(processed_ids)}"
    )

    print(
        f"Remaining: "
        f"{len(remaining_df)}"
    )


    for _, row in tqdm(
        remaining_df.iterrows(),
        total=len(remaining_df),
        desc=SELECTED_MODEL,
    ):

        response = generate_single_response(
            client=client,
            context=row["context"],
            question=row["question"],
        )


        record = {
            "question_id": str(
                row["question_id"]
            ),

            "model_label": (
                SELECTED_MODEL
            ),

            "model_id": (
                MODEL_ID
            ),

            "provider": (
                PROVIDER
            ),

            "model_reasoning": (
                response["reasoning"]
            ),

            "model_answer": (
                response["answer"]
            ),

            "model_confidence": (
                response["confidence"]
            ),

            "raw_response": (
                response["raw_response"]
            ),

            "error": (
                response["error"]
            ),
        }


        records.append(
            record
        )


        if (
            len(records)
            % CHECKPOINT_EVERY
            == 0
        ):

            checkpoint_payload = {
                "metadata": {
                    **metadata,
                    "created_at": created_at,
                    "updated_at": utc_now(),
                    "status": "in_progress",
                },

                "records": records,
            }


            save_json_atomic(
                checkpoint_payload,
                CHECKPOINT_FILE,
            )


        if REQUEST_DELAY_SECONDS > 0:
            time.sleep(
                REQUEST_DELAY_SECONDS
            )


    final_metadata = {
        **metadata,

        "created_at": (
            created_at
        ),

        "completed_at": (
            utc_now()
        ),

        "status": "completed",

        "record_count": (
            len(records)
        ),
    }


    final_payload = {
        "metadata": final_metadata,
        "records": records,
    }


    save_json_atomic(
        final_payload,
        FINAL_OUTPUT_FILE,
    )


    save_json_atomic(
        final_metadata,
        METADATA_FILE,
    )


    # Keep the final checkpoint synchronized.
    save_json_atomic(
        final_payload,
        CHECKPOINT_FILE,
    )


    print(
        "\n✓ Generation completed."
    )

    print(
        f"Records: {len(records)}"
    )

    print(
        f"Output: {FINAL_OUTPUT_FILE}"
    )


    return pd.DataFrame(
        records
    )

## 15. Generation switch

Running the following cell does **not** make API requests while
`RUN_GENERATION = False`.

To intentionally start a generation run, review the configuration above and
manually change it to `True`.

In [46]:
generation_df = None


if RUN_GENERATION:

    print(
        "Generation is ENABLED."
    )

    print(
        f"Model: {MODEL_ID}"
    )

    print(
        f"Questions: {len(df)}"
    )

    generation_df = (
        run_generation()
    )


else:

    print(
        "Generation is disabled."
    )

    print(
        "No API request was sent."
    )

    print(
        "Set RUN_GENERATION = True "
        "only when you intentionally "
        "want to run the experiment."
    )

Generation is ENABLED.
Model: gpt-4o-mini
Questions: 5
Already processed: 0
Remaining: 5


gpt-4o-mini:   0%|          | 0/5 [00:00<?, ?it/s]


✓ Generation completed.
Records: 5
Output: /content/drive/MyDrive/turkish-llm-reasoning/results/raw/base/smoke__gpt-4o-mini__smoke_v1.json


## 16. End of generation notebook

This notebook intentionally ends after raw model generation.

It does not calculate:

- answer accuracy,
- faithfulness,
- calibration,
- hallucination-risk metrics,
- statistical comparisons,
- figures.

Those stages are separated from generation so that saved model responses can
be re-evaluated without repeating potentially expensive API calls.

The next experiment notebook is:

`02_closed_models_variants.ipynb`